In [1]:
import os, sys
os.environ['CUDA_VISIBLE_DEVICES'] = '1' # set this first before importing torch

import torch  
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Number of GPUs visible: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}: {torch.cuda.get_device_name(i)}")

from torch.utils.data import DataLoader, TensorDataset      
import numpy as np
import pandas as pd
import xarray as xr
import zarr
import fsspec
import nc_time_axis
from functools import partial
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from scipy import stats
import warnings
warnings.filterwarnings('ignore')
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import time
from tqdm import tqdm
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
import xesmf as xe
import joblib
from scipy import stats
import copy
from pathlib import Path
import joblib

PyTorch version: 2.10.0+cu126
CUDA available: True
Number of GPUs visible: 1
GPU 0: NVIDIA A100-SXM4-80GB MIG 1g.10gb


## CMIP6 data access using Google Cloud Storage

In [7]:
# ---------------------------------------------------------------------
# 1. Configuration
# ---------------------------------------------------------------------
CATALOG_URL   = "https://storage.googleapis.com/cmip6/pangeo-cmip6.csv"
source_id     = "MPI-ESM1-2-HR"
table_id      = "Amon"
experiment_id = "historical"
member_id     = "r1i1p1f1" 
grid_label    = ['gn', 'gr']
clim_var      = ['rsds', 'clt', 'tas','ua','va','hur','zg','hurs']
 
lon_bnds = (85, 150)                
lat_bnds = (-15, 10)
baseline = slice("1970-01-01", "2014-12-31") # 45 years of monthly data  
 
df = pd.read_csv(CATALOG_URL, low_memory=False)

# ---------------------------------------------------------------------
# 2. Helper: crop domain (handle inconsistency of longitude 0-360 vs -180..180,
#    latitude)
# ---------------------------------------------------------------------
def crop_domain(ds: xr.Dataset, lon_bnds, lat_bnds) -> xr.Dataset:
    lon_min, lon_max = lon_bnds
    if ds["lon"].max() <= 180:
        lon_min = ((lon_min + 180) % 360) - 180
        lon_max = ((lon_max + 180) % 360) - 180
 
    lat_descending = bool(ds["lat"][0] > ds["lat"][-1])
    lat_slice = (slice(max(lat_bnds), min(lat_bnds)) if lat_descending
                 else slice(min(lat_bnds), max(lat_bnds)))
 
    return ds.sel(lon=slice(lon_min, lon_max), lat=lat_slice)
 
def sanitize_time(ds: xr.Dataset) -> xr.Dataset:
    """Ensure time coordinate is sorted.
    """
    time_index = ds.indexes["time"]
 
    if not time_index.is_monotonic_increasing:
        print("    [!] time is not sorted -- re-sorted time index")
        ds = ds.sortby("time")
        time_index = ds.indexes["time"]
 
    if not time_index.is_unique:
        n_dupes = len(time_index) - len(np.unique(time_index.values))
        print(f"    [!] {n_dupes} remove time index duplication")
        _, unique_pos = np.unique(time_index.values, return_index=True)
        ds = ds.isel(time=np.sort(unique_pos))
 
    return ds
 
# ---------------------------------------------------------------------
# 3. Loop per variabel, crop the domain and store in a dict
# ---------------------------------------------------------------------
selected_data = {}
missing_vars  = []
 
print(f"Accessing {source_id} / {experiment_id} / {table_id} for {len(clim_var)} variabel...\n")
 
for var in clim_var:
    match = df.query(
        "source_id == @source_id "
        "& table_id == @table_id "
        "& variable_id == @var "
        "& experiment_id == @experiment_id "
        "& grid_label == @grid_label "
        "& member_id == @member_id"
    )
 
    if match.empty:
        print(f"[SKIP] {var:<10} not found for {source_id}/{member_id}")
        missing_vars.append(var)
        continue
 
    if "version" in match.columns:
        match = match.sort_values("version")
    row = match.iloc[-1]
 
    try:
        ds = xr.open_zarr(
            row.zstore,
            storage_options={"token": "anon"},
            consolidated=True,
        )
        ds_cropped = crop_domain(ds, lon_bnds, lat_bnds)
        ds_cropped = sanitize_time(ds_cropped)
 
        da = ds_cropped[var].sel(time=baseline)
 
        # Validate: Make sure it is exactly 540 months (January 1970 through December 2014), with no gaps
        if da.sizes["time"] != 540:
            print(f"    [!] time step = {da.sizes['time']}, is different -- "
                  f"there is possibility of data loss from source!")
 
        selected_data[var] = da
        print(f"[OK]   {var:<10} shape={da.shape}  "
              f"period={da.time.values[0]}..{da.time.values[-1]}")
    except Exception as e:
        print(f"[ERROR] {var:<10} faild to open: {type(e).__name__}: {e}")
        
        try:
            ti = ds.indexes["time"]
            cal = getattr(ti, "calendar", "standard (datetime64)")
            print(f"        diagnostic: calendar={cal}, "
                  f"monotonic={ti.is_monotonic_increasing}, unique={ti.is_unique}, "
                  f"range=({ti[0]} .. {ti[-1]})")
        except Exception:
            pass
        missing_vars.append(var)
 
# ---------------------------------------------------------------------
# 4. Summary
# ---------------------------------------------------------------------
print(f"\n{'='*55}")
print(f"{len(selected_data)}/{len(clim_var)} varaible is cropped.")
if missing_vars:
    print(f"Problematic variable: {missing_vars}")
print(f"{'='*55}")

Accessing MPI-ESM1-2-HR / historical / Amon for 8 variabel...

[OK]   rsds       shape=(540, 27, 70)  period=1970-01-16T12:00:00.000000000..2014-12-16T12:00:00.000000000
[OK]   clt        shape=(540, 27, 70)  period=1970-01-16T12:00:00.000000000..2014-12-16T12:00:00.000000000
    [!] time is not sorted -- re-sorted time index
[OK]   tas        shape=(540, 27, 70)  period=1970-01-16T12:00:00.000000000..2014-12-16T12:00:00.000000000
[OK]   ua         shape=(540, 19, 27, 70)  period=1970-01-16T12:00:00.000000000..2014-12-16T12:00:00.000000000
[OK]   va         shape=(540, 19, 27, 70)  period=1970-01-16T12:00:00.000000000..2014-12-16T12:00:00.000000000
[OK]   hur        shape=(540, 19, 27, 70)  period=1970-01-16T12:00:00.000000000..2014-12-16T12:00:00.000000000
[OK]   zg         shape=(540, 19, 27, 70)  period=1970-01-16T12:00:00.000000000..2014-12-16T12:00:00.000000000
[OK]   hurs       shape=(540, 27, 70)  period=1970-01-16T12:00:00.000000000..2014-12-16T12:00:00.000000000

8/8 varaible 

## Select climate predictors based on levels

In [3]:
# Define pressure levels to extract
plev_levels = [85000, 70000, 50000]

# Variables that have a plev dimension
plev_vars = ['hur', 'ua', 'va', 'zg']

# Create a new dictionary for the selected data
new_selected_data = {}

# Variables without plev dimension
no_plev_vars = ['tas', 'hurs', 'rsds', 'clt']

# Add pressure-level variables with level in key name
for var in plev_vars:
    if var in selected_data:
        for level in plev_levels:
            key_name = f"{var}_{level}"  # e.g., 'ta_85000', 'ua_70000'
            new_selected_data[key_name] = selected_data[var].sel(plev=level)

# Add non-plev variables (no level suffix)
for var in no_plev_vars:
    if var in selected_data:
        new_selected_data[var] = selected_data[var]

# Remove variables with low correlation with target data (based on previous analysis)
keys_to_remove = ['zg_70000', 'zg_50000', 'va_70000', 'va_50000', 
                  'ua_70000', 'ua_50000']

for key in keys_to_remove:
    if key in new_selected_data:  # Check if exists to avoid KeyError
        del new_selected_data[key]

In [4]:
new_selected_data.keys()

dict_keys(['hur_85000', 'hur_70000', 'hur_50000', 'ua_85000', 'va_85000', 'zg_85000', 'tas', 'hurs', 'rsds', 'clt'])

## Apply PCA analysis using areal averaged approach

In [6]:
def area_weighted_average(da, key):
    """
    Calculate area-weighted spatial mean → 1 values per timestep.
    The latitude cosine weighting factor decreases as the grid cell area shrinks toward the poles.

    Input  : DataArray [time × lat × lon]
    Output : Series    [time]
    """
    weights = np.cos(np.deg2rad(da.lat))
    da_avg  = da.weighted(weights).mean(dim=['lat', 'lon'])

    return pd.Series(
        da_avg.values,
        index=pd.to_datetime(da.time.values),
        name=key
    )

def create_predictor_matrix(selected_data):
    """
    Convert DataArray [time×lat×lon] to
    DataFrame [n_time × n_variabel] via area-weighted average.

    """
    print("Calculate area-weighted average...")
    print(f"  {'Key':<15} {'Shape input':>16} {'→ output':>12}")
    print("  " + "─"*45)

    series_list = []
    for key, da in selected_data.items():
        s = area_weighted_average(da, key)
        series_list.append(s)
        print(f"  {key:<15} {str(da.shape):>16} {'→ ('+str(len(s))+',)':>12}")

    X = pd.concat(series_list, axis=1)
    X.index.name = 'time'

    print(f"\n✓ Predictor matrix X:")
    print(f"  Shape  : {X.shape}  → [n_month × n_variable]")
    print(f"  Period: {X.index[0].date()} s/d {X.index[-1].date()}")
    print(f"  Missing: {X.isna().sum().sum()}")
    return X

# Create matrix [ntime_ 360 × n_climate ]
X_area = create_predictor_matrix(new_selected_data)

# ════════════════════════════════════════════════════════════════
# PREPROCESSING
# ════════════════════════════════════════════════════════════════
def preprocessing_final(X, method_impute='linear'):
    """
    Preprocessing before apply Principal Component Analysis (PCA).
    Apply standardization here.
    """
    X_proc = X.copy()

    # ── Handle missing values ──────────────────────────────────────
    n_missing = X_proc.isna().sum().sum()
    if n_missing > 0:
        print(f"Handling {n_missing} missing values "
              f"(metode: {method_impute})...")
        if method_impute == 'linear':
            X_proc = X_proc.interpolate(method='time')
            X_proc = X_proc.ffill().bfill()
        elif method_impute == 'mean':
            imputer = SimpleImputer(strategy='mean')
            X_proc  = pd.DataFrame(
                imputer.fit_transform(X_proc),
                index=X_proc.index,
                columns=X_proc.columns
            )

    # ── Standardize the data — once here ───────────────────────────────
    scaler   = StandardScaler()
    X_scaled = scaler.fit_transform(X_proc)
    X_scaled = pd.DataFrame(X_scaled,
                             index=X_proc.index,
                             columns=X_proc.columns)

    print(f"\n✓ Preprocessing is done")
    print(f"  Shape    : {X_scaled.shape}")
    print(f"  Mean ≈ 0 : {X_scaled.mean().mean():.6f}")
    print(f"  Std  ≈ 1 : {X_scaled.std().mean():.6f}")

    return X_scaled, scaler

X_ready, scaler_fitted = preprocessing_final(X_area)

print("\n" + "═"*60)
print("✅ Matrix X is ready for PCA!")
print(f"   Shape : {X_ready.shape}  → [n_month × n_variable]")
print(f"   Condition: n_time={X_ready.shape[0]} > n_var={X_ready.shape[1]}"
      f"  → well-conditioned ✓")
print("═"*60)

# ════════════════════════════════════════════════════════════════
# PCA
# ════════════════════════════════════════════════════════════════
# Apply full PCA for diagnostic
pca_full = PCA()          # only fit 
pca_full.fit(X_ready)     # ← fit() once here

eigenvalues = pca_full.explained_variance_
evr         = pca_full.explained_variance_ratio_
cumulative  = np.cumsum(evr)
n_total     = len(eigenvalues)

variance_thresholds = [0.80, 0.90, 0.95]
pc_at_threshold = {
    thr: int(np.argmax(cumulative >= thr) + 1)
    for thr in variance_thresholds
}

# Diagnostic summary
print("\n" + "═"*55)
print(f"{'PCA DIAGNOSTICS':^55}")
print("═"*55)
print(f"  Input shape    : {X_ready.shape}")
print(f"  Total PC       : {n_total}  (= n_variable)")
print(f"  Max PC possible: {n_total}  ← well-conditioned!")
print(f"\n  {'PC':<6} {'Eigenvalue':>11} {'Var%':>7} {'Cum%':>9}")
print(f"  {'──':<6} {'──────────':>11} {'────':>7} {'────':>9}")
for i in range(n_total):
    marker = ""
    for thr in variance_thresholds:
        if pc_at_threshold[thr] == i + 1:
            marker += f"  ← {thr*100:.0f}%"
    print(f"  PC{i+1:<4} {eigenvalues[i]:>11.4f} "
          f"{evr[i]*100:>7.3f} {cumulative[i]*100:>9.3f}{marker}")
print("═"*55)
print(f"\n  pc_at_threshold = {pc_at_threshold}")

# Use optimal PC (90%) for downscaling purpose
n_pc_optimal = pc_at_threshold[0.90]
pca_optimal  = PCA(n_components=n_pc_optimal)
PC_scores    = pca_optimal.fit_transform(X_ready)  # ← use fit_transform here

#print(f"\n✅ PC scores for downscaling: {PC_scores.shape}")
#print(f"   {n_pc_optimal} PC merepresentasikan 95% variansi prediktor")
#n_pc_optimal = 6   # ← maximum 
#pca_optimal  = PCA(n_components=n_pc_optimal)
#PC_scores    = pca_optimal.fit_transform(X_ready)

print(f"✓ PC scores shape: {PC_scores.shape}")
print(f"  [{PC_scores.shape[0]} bulan × {PC_scores.shape[1]} PC]")

Calculate area-weighted average...
  Key                  Shape input     → output
  ─────────────────────────────────────────────
  hur_85000          (540, 27, 70)     → (540,)
  hur_70000          (540, 27, 70)     → (540,)
  hur_50000          (540, 27, 70)     → (540,)
  ua_85000           (540, 27, 70)     → (540,)
  va_85000           (540, 27, 70)     → (540,)
  zg_85000           (540, 27, 70)     → (540,)
  tas                (540, 27, 70)     → (540,)
  hurs               (540, 27, 70)     → (540,)
  rsds               (540, 27, 70)     → (540,)
  clt                (540, 27, 70)     → (540,)

✓ Predictor matrix X:
  Shape  : (540, 10)  → [n_month × n_variable]
  Period: 1970-01-16 s/d 2014-12-16
  Missing: 0

✓ Preprocessing is done
  Shape    : (540, 10)
  Mean ≈ 0 : 0.000000
  Std  ≈ 1 : 1.000927

════════════════════════════════════════════════════════════
✅ Matrix X is ready for PCA!
   Shape : (540, 10)  → [n_month × n_variable]
   Condition: n_time=540 > n_var=10  → w

## Pre-processing ANN predictor

In [8]:
# Read another predictor and target data
PATHS = {
    'OBS': "/raid/nfs_storageIPB/thesis_JL/data/interim/historical_dataset/",
    'DEM': "/raid/nfs_storageIPB/thesis_JL/data/interim/"
}

TARGET_FILENAME = "era5land_derived_PVOut_potential_JAVA_19700101-20211231.nc"
DEM_FILENAME = "SRTM_regrided_conservative_0p1deg_Indonesia.nc"

# Read target data
target_ds = xr.open_dataset(PATHS['OBS']+TARGET_FILENAME)
# Calculate monthly mean 
PV_mean = target_ds.pv_output_potential.resample(valid_time='1ME').mean()
sel_PV_mean = PV_mean.sel(valid_time=slice('1970','2014')) 
# Rename reference data dimension for consistency
PV_target = sel_PV_mean.rename({
    'valid_time': 'time',
    'latitude': 'lat', 
    'longitude': 'lon'
})

# Read SRTM data
DEM_ds = xr.open_dataset(PATHS['DEM']+DEM_FILENAME)
lon_obs,lat_obs  = (103, 117),(-9.5, -4.5) 
sel_DEM_ds = DEM_ds.sel(**{'longitude': slice(*lon_obs),
                             'latitude': slice(lat_obs[1], lat_obs[0])}
                         ) 
# Rename dimension of elevation for consistency
sel_DEM_renamed = sel_DEM_ds.rename({
    'latitude': 'lat', 
    'longitude': 'lon'}
 )

DEM_da = xr.DataArray(
    np.tile(sel_DEM_renamed.elevation.values, (len(PV_target.time), 1, 1)),
    dims=['time', 'lat', 'lon'],
    coords={
        'time': PV_target.time,
        'lat': PV_target.lat,
        'lon': PV_target.lon
    }
)

# ── 1. PCs ───────────────────────────────────────────────────────────────────
# Covert PC_scores to xarray.dataarray with time dimension
# PC_scores shape: (360, n_pc_optimal) - [time × PCs]
PC_scores_da = xr.DataArray(
    PC_scores,
    dims=['time', 'pc'],
    coords={
        'time': PV_target.time,  # Use same time coordinates
        'pc': [f'PC{i+1}' for i in range(PC_scores.shape[1])]
    },
    name='PC_scores'
)

print(f"PC_scores_da shape: {PC_scores_da.shape}")
print(f"PC_scores_da dims: {PC_scores_da.dims}")

# ── 2. LAT, LON, MONTH ───────────────────────────────────────────────────────────────────
# Get months of the year (1-12)
month_of_year = PV_target.time.dt.month

# Broadcast latxlonxtime to match target data
moy_da = month_of_year.broadcast_like(PV_target)
lat_da = PV_target.lat.broadcast_like(PV_target)
lon_da = PV_target.lon.broadcast_like(PV_target)
#DEM_da = DEM_da.broadcast_like(PV_target)
PC_da = PC_scores_da.broadcast_like(PV_target)

# Create dictionary with each PC as a DataArray
PC_dict = {str(pc): PC_da.sel(pc=pc) for pc in PC_da.pc.values}
pc_name = PC_da.pc.values

PC_scores_da shape: (540, 5)
PC_scores_da dims: ('time', 'pc')


In [10]:
#DEM_da[100,:,:].plot()

In [11]:
# ════════════════════════════════════════════════════════════════════════════
# Step 1: Create a function to extract all data without scaling
# ════════════════════════════════════════════════════════════════════════════
def extract_raw_data(pc_dict, pc_name, pv_da, lon_da, lat_da,
                     month_da, DEM_da, start_year=1970, end_year=2002):
    """
    Preparing dataset with time filtering.
    Extract and flatten data without scaling.
    
    Parameters
    ----------
    pc_dict   : dict — {pc_name: xr.DataArray [time×lat×lon]}
    pc_name   : list — PCs ['PC1','PC2',...]
    pv_da     : xr.DataArray [time×lat×lon] — target PV potential
    lon_da    : xr.DataArray [time×lat×lon] — longitude (broadcast)
    lat_da    : xr.DataArray [time×lat×lon] — latitude (broadcast)
    month_da  : xr.DataArray [time×lat×lon] — month 1–12 (broadcast)
    DEM_da    : xr.DataArray [time×lat×lon] — elevation (broadcast)
    start_year: int 
    end_year  : int

    Returns X_2d, y_2d, valid_mask, grid_indices (all NaN-removed).
 
    grid_indices[i] = which spatial grid point sample i belongs to,
    indexed into the flattened (n_lat × n_lon) grid.

    """
    # Get time dimension from any climate variable
    sample_var = pc_dict[pc_name[0]]
    
    # Extract year from time coordinate
    if hasattr(sample_var.time, 'dt'):
        years = sample_var.time.dt.year.values
    else:
        # If time is not a datetime, try to extract from string representation
        years = np.array([pd.to_datetime(t).year for t in sample_var.time.values])
    
    # Create mask for years 1985-2008
    time_mask = (years >= start_year) & (years <= end_year)
    
    print(f"\nFiltering time dimension: {start_year} to {end_year}")
    print(f"Original time points: {len(years)}")
    print(f"Selected time points: {time_mask.sum()}")

    # Filter all PC based on period
    pc_arrays = []
    for var in pc_name:
        # Select time subset using mask
        da_subset = pc_dict[var].isel(time=time_mask)
        pc_arrays.append(da_subset.values)

    # Stack PC (time, lat, lon, n_features)
    pc_stack = np.stack(pc_arrays, axis=-1)
    
    # Filter other 3D variable 
    month_array = month_da.isel(time=time_mask).values  
    sin_array = np.sin(2 * np.pi * month_array / 12) 
    cos_array = np.cos(2 * np.pi * month_array / 12)
    
    lat_array = lat_da.isel(time=time_mask).values  
    lon_array = lon_da.isel(time=time_mask).values   
    elevation_array = DEM_da.isel(time=time_mask).values
    pv_array        = pv_da.isel(time=time_mask).values
    
    # Get dimensions
    n_time, n_lat, n_lon, n_pc = pc_stack.shape
    n_grid = n_lat * n_lon
    
    print(f"\nData dimensions after filtering:")
    print(f"  Time points: {n_time}")
    print(f"  Grid points: {n_grid} ({n_lat} × {n_lon})")
    print(f"  Climate features: {n_pc}")

    # Flatten spatial dimensions for all variables
    pc_flat         = pc_stack.reshape(n_time, n_grid, n_pc)
    lat_flat        = lat_array.reshape(n_time, n_grid)
    lon_flat        = lon_array.reshape(n_time, n_grid)
    sin_month_flat  = sin_array.reshape(n_time, n_grid)
    cos_month_flat  = cos_array.reshape(n_time, n_grid)
    elevation_flat  = elevation_array.reshape(n_time, n_grid)

    n_features = n_pc + 5  # number of PC + lon,lat,elev,sin,cos
    X_3d = np.zeros((n_time, n_grid, n_features), dtype=np.float32)
    
    X_3d[:, :, :n_pc]     = pc_flat
    X_3d[:, :, n_pc]      = lon_flat
    X_3d[:, :, n_pc + 1]  = lat_flat
    X_3d[:, :, n_pc + 2]  = elevation_flat
    X_3d[:, :, n_pc + 3]  = sin_month_flat
    X_3d[:, :, n_pc + 4]  = cos_month_flat

    X_2d = X_3d.reshape(-1, n_features)
    y_2d = pv_array.reshape(-1)

    # ── grid index for each sample before NaN removal ──
    # grid_indices[i] tells which of the n_grid spatial points sample i is from.
    # np.tile([0,1,...,n_grid-1], n_time) → length = n_time * n_grid
    grid_indices_flat = np.tile(np.arange(n_grid), n_time)
    
    # Remove NaN
    valid_mask = ~np.isnan(X_2d).any(axis=1) & ~np.isnan(y_2d)
    
    return (X_2d[valid_mask],
            y_2d[valid_mask],
            valid_mask,
            grid_indices_flat[valid_mask])   # ← new 4th return value

# Usage - training set (1970-2002, 33 years) 
X_train, y_train, valid_mask_train, grid_idx_train = extract_raw_data(
    PC_dict, pc_name, PV_target, lon_da, lat_da,
    moy_da, DEM_da, start_year=1970, end_year=2002)
X_val,   y_val,   valid_mask_val,   grid_idx_val   = extract_raw_data(
    PC_dict, pc_name, PV_target, lon_da, lat_da,
    moy_da, DEM_da, start_year=2003, end_year=2008)
X_test,  y_test,  valid_mask_test,  grid_idx_test  = extract_raw_data(
    PC_dict, pc_name, PV_target, lon_da, lat_da,
    moy_da, DEM_da, start_year=2009, end_year=2014)
print(f"\nShapes → X_train:{X_train.shape}  y_train:{y_train.shape}")
print(f"Features per sample: {X_train.shape[1]}")


Filtering time dimension: 1970 to 2002
Original time points: 540
Selected time points: 396

Data dimensions after filtering:
  Time points: 396
  Grid points: 7000 (50 × 140)
  Climate features: 5

Filtering time dimension: 2003 to 2008
Original time points: 540
Selected time points: 72

Data dimensions after filtering:
  Time points: 72
  Grid points: 7000 (50 × 140)
  Climate features: 5

Filtering time dimension: 2009 to 2014
Original time points: 540
Selected time points: 72

Data dimensions after filtering:
  Time points: 72
  Grid points: 7000 (50 × 140)
  Climate features: 5

Shapes → X_train:(550836, 10)  y_train:(550836,)
Features per sample: 10


In [12]:
# ════════════════════════════════════════════════════════════════
# Scaling process
# ════════════════════════════════════════════════════════════════

class PCMonthlyScaler:
    """
    A scaler specifically designed for monthly PC features.

    Philosophy:
      PC scores   → StandardScaler  (normal distribution from PCA)
      lon, lat    → MinMaxScaler    (coordinates, defined range)
      elevation   → RobustScaler   (skewed, mountain outliers present)
      sin, cos    → NOT scaled  (already bounded [-1, 1])
      Y (PV)      → MinMaxScaler   (non-negative, clear physical range)

    Fit ONLY on the training set → transform all splits..

    Parameters
    ----------
    n_pc : int — the sum of the PC scores in X
    """

    def __init__(self, n_pc):
        self.n_pc = n_pc

        # Column index
        self.idx = {
            'pc'  : list(range(n_pc)),    # 0 … n_pc-1
            'lon' : [n_pc],               # n_pc
            'lat' : [n_pc + 1],           # n_pc+1
            'elev': [n_pc + 2],           # n_pc+2
            'sin' : [n_pc + 3],           # n_pc+3 → not scaled
            'cos' : [n_pc + 4],           # n_pc+4 → not scaled
        }

        self.scalers = {
            'pc'  : StandardScaler(),
            'lon' : MinMaxScaler(feature_range=(-1, 1)),
            'lat' : MinMaxScaler(feature_range=(-1, 1)),
            'elev': RobustScaler(),
        }
        # sin and cos do not require a scaler
        self.scaler_y = MinMaxScaler(feature_range=(0, 1))
        self.fitted   = False

    def fit(self, X, y):
        """Fits ONLY the training data."""
        for grup, scaler in self.scalers.items():
            scaler.fit(X[:, self.idx[grup]])

        self.scaler_y.fit(y.reshape(-1, 1))
        self.fitted = True

        print("✓ PCMonthlyScaler fitted at training set")
        print(f"  n_PC    : {self.n_pc}")
        print(f"  n_sampel: {len(X):,}")
        self._print_stats(X, y)
        return self

    def transform_X(self, X):
        """Transform feature X — sin/cos left unchanged."""
        X_s = X.copy().astype(np.float32)
        for grup, scaler in self.scalers.items():
            X_s[:, self.idx[grup]] = scaler.transform(
                X[:, self.idx[grup]]
            )
        return X_s

    def transform_y(self, y):
        return self.scaler_y.transform(
            y.reshape(-1, 1)
        ).flatten()

    def inverse_transform_y(self, y_scaled):
        return self.scaler_y.inverse_transform(
            y_scaled.reshape(-1, 1)
        ).flatten()

    def fit_transform_X(self, X):
        return self.fit_transform_helper(X)

    def _print_stats(self, X, y):
        """Print statistics after scaling."""
        X_s = self.transform_X(X)
        y_s = self.transform_y(y)

        n_pc = self.n_pc
        print(f"\n  Verify the range after scaling:")
        print(f"  {'Grup':<12} {'min':>8} {'max':>8} {'mean':>8} {'std':>8}")
        print("  " + "─"*44)

        grup_check = {
            'PC scores': X_s[:, :n_pc].flatten(),
            'lon'      : X_s[:, n_pc],
            'lat'      : X_s[:, n_pc+1],
            'elevation': X_s[:, n_pc+2],
            'sin_month': X_s[:, n_pc+3],
            'cos_month': X_s[:, n_pc+4],
            'Y (PV)'   : y_s,
        }
        for nama, vals in grup_check.items():
            print(f"  {nama:<12} {vals.min():>8.3f} {vals.max():>8.3f} "
                  f"{vals.mean():>8.3f} {vals.std():>8.3f}")

    def save(self, path):
        joblib.dump(self, path)
        print(f"✓ Scaler saved: {path}")

    @staticmethod
    def load(path):
        return joblib.load(path)


# ════════════════════════════════════════════════════════════════
# EXECUTION
# ════════════════════════════════════════════════════════════════

n_pc = len(pc_name)

# Fits ONLY on the training set
scaler = PCMonthlyScaler(n_pc=n_pc)
scaler.fit(X_train, y_train)

# Transform all splits using the same scaler
X_train_sc = scaler.transform_X(X_train)
X_val_sc   = scaler.transform_X(X_val)
X_test_sc  = scaler.transform_X(X_test)

y_train_sc = scaler.transform_y(y_train)
y_val_sc   = scaler.transform_y(y_val)
y_test_sc  = scaler.transform_y(y_test)

# Simpan scaler 
scaler_dir = f"/raid/nfs_storageIPB/thesis_JL/data/ANN_scaler"
scaler.save(f"{scaler_dir}/{source_id}_data_scaler.pkl")

# ── Final summary ───────────────────────────────────────────
print("\n" + "═"*55)
print("DATASET READY FOR ANN")
print("═"*55)
print(f"  {'Split':<8} {'X shape':>20} {'y shape':>14}")
print(f"  {'─────':<8} {'───────':>20} {'───────':>14}")
for name, X, y in [('Train', X_train_sc, y_train_sc),
                    ('Val',   X_val_sc,   y_val_sc),
                    ('Test',  X_test_sc,  y_test_sc)]:
    print(f"  {name:<8} {str(X.shape):>20} {str(y.shape):>14}")

✓ PCMonthlyScaler fitted at training set
  n_PC    : 5
  n_sampel: 550,836

  Verify the range after scaling:
  Grup              min      max     mean      std
  ────────────────────────────────────────────
  PC scores      -3.334    3.097    0.000    1.000
  lon            -1.000    1.000   -0.076    0.487
  lat            -1.000    1.000   -0.088    0.481
  elevation      -0.399    4.733    0.380    0.910
  sin_month      -1.000    1.000    0.000    0.707
  cos_month      -1.000    1.000    0.000    0.707
  Y (PV)          0.000    1.000    0.448    0.149
✓ Scaler saved: /raid/nfs_storageIPB/thesis_JL/data/ANN_scaler/MPI-ESM1-2-HR_data_scaler.pkl

═══════════════════════════════════════════════════════
DATASET READY FOR ANN
═══════════════════════════════════════════════════════
  Split                 X shape        y shape
  ─────                 ───────        ───────
  Train            (550836, 10)      (550836,)
  Val              (100152, 10)      (100152,)
  Test             

## ANN Model Training

In [13]:
# ════════════════════════════════════════════════════════════════════════════
# DATALOADER BUILDER 
# ════════════════════════════════════════════════════════════════════════════
torch.manual_seed(42)  
batch_size = 2048 #4096 #3072 #2048  #256 #1024  

def make_loaders(X_tr, y_tr, X_v, y_v, X_te, y_te):
    """
    Create DataLoaders for each set
    """
 
    def _dataset(X, y):
        tensors = [torch.FloatTensor(X), torch.FloatTensor(y)]
        return TensorDataset(*tensors)
 
    loader_cfg = dict(num_workers=4, pin_memory=True, prefetch_factor=4)
    return (
        DataLoader(_dataset(X_tr, y_tr), batch_size=batch_size,
                   shuffle=True,  **loader_cfg),
        DataLoader(_dataset(X_v,  y_v),  batch_size=batch_size,
                   shuffle=False, **loader_cfg),
        DataLoader(_dataset(X_te, y_te), batch_size=batch_size,
                   shuffle=False, **loader_cfg),
    )
 
train_loader, val_loader, test_loader = make_loaders(
    X_train_sc, y_train_sc, X_val_sc, y_val_sc, X_test_sc, y_test_sc)

In [14]:
# ════════════════════════════════════════════════════════════════════════════
# MODEL INSTANTIATION
# ════════════════════════════════════════════════════════════════════════════
num_nodes = '32_16'
h1=32
h2=16
dropout_rate = 0.2

class PVDownscaling(nn.Module):
    def __init__(self, in_features=X_train.shape[1], h1=h1, h2=h2, out_features=1, p_drop=dropout_rate): # in_features=9
        super().__init__()
        self.fc1 = nn.Linear(in_features, h1)
        self.bn1 = nn.BatchNorm1d(h1)
        self.drop1 = nn.Dropout(p_drop)
        
        self.fc2 = nn.Linear(h1, h2)
        self.bn2 = nn.BatchNorm1d(h2)
        self.drop2 = nn.Dropout(p_drop)
        
        self.fc3 = nn.Linear(h2, out_features)
        
    def forward(self, x):
        # Linear -> BN -> ReLU -> Dropout
        x = self.drop1(F.relu(self.bn1(self.fc1(x)))) 
        x = self.drop2(F.relu(self.bn2(self.fc2(x))))
        x = self.fc3(x)
        return x

# Build arch_str ONCE; reuse for checkpoint and metrics file
arch_str = f"{X_train.shape[1]}→{num_nodes}→1"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
 
# Instantiate two independent models (one per scaling option for comparison)
model = PVDownscaling(in_features=X_train.shape[1],
                      h1=h1, h2=h2, out_features=1, p_drop=dropout_rate).to(device)

Using device: cuda


In [15]:
# ════════════════════════════════════════════════════════════════════════════
# TRAINING LOOP 
# ════════════════════════════════════════════════════════════════════════════
LR = 3e-4
def train_model(model, train_loader, val_loader,
                save_path, epochs=150, patience=30, min_delta=1e-5,
                lr=LR): # weight_decay=1e-5
    """
    Returns (trained_model, train_losses, val_losses, best_epoch, best_state).
    Stored in checkpoint metadata.
    """
    criterion  = nn.MSELoss()
    optimizer  = torch.optim.Adam(model.parameters(), lr=lr) # weight_decay=1e-4
    scheduler  = torch.optim.lr_scheduler.ReduceLROnPlateau(
                     optimizer, mode='min', factor=0.5,
                     patience=7, min_lr=1e-6)
  
    best_val_loss  = float('inf')
    best_epoch     = 0
    patience_counter = 0
    best_state     = None
    train_losses, val_losses, lr_history = [], [], []
 
    # Detect whether loader yields 2 or 3 tensors
    sample_batch = next(iter(train_loader))
    has_grid_idx = len(sample_batch) == 3
 
    print(f"\n{'─'*60}")
    print(f"Training ANN model arch={arch_str} ")
    print(f"Early stopping: patience={patience}, min_delta={min_delta}")
    print(f"{'─'*60}")
 
    for epoch in range(epochs):
        t0 = time.time()
        model.train()
        total_train, n_train = 0.0, 0
 
        for batch in tqdm(train_loader,
                          desc=f'Ep {epoch+1}/{epochs} [Train]',
                          leave=False):
            bX = batch[0].to(device)
            by = batch[1].to(device)
            pred = model(bX)
            loss = criterion(pred.squeeze(), by)
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            total_train += loss.item(); n_train += 1
 
        avg_train = total_train / n_train
 
        model.eval()
        total_val, n_val = 0.0, 0
        with torch.no_grad():
            for batch in tqdm(val_loader,
                              desc=f'Ep {epoch+1}/{epochs} [Val]',
                              leave=False):
                bX = batch[0].to(device)
                by = batch[1].to(device)
                vl = criterion(model(bX).squeeze(), by)
                total_val += vl.item(); n_val += 1
 
        avg_val    = total_val / n_val
        current_lr = optimizer.param_groups[0]['lr']
        train_losses.append(avg_train)
        val_losses  .append(avg_val)
        lr_history  .append(current_lr)
 
        # ── early stopping ───────────────────────────────────────────────
        if avg_val < best_val_loss - min_delta:
            improvement   = best_val_loss - avg_val
            best_val_loss = avg_val
            best_epoch    = epoch
            patience_counter = 0
            best_state = copy.deepcopy(model.state_dict())
 
            # ── dynamic metadata in checkpoint ───────────────────
            torch.save({
                'epoch'              : epoch,
                'model_state_dict'   : best_state,
                'optimizer_state_dict': optimizer.state_dict(),
                'train_loss'         : avg_train,
                'val_loss'           : avg_val,
                'train_losses'       : train_losses,
                'val_losses'         : val_losses,
                'lr_history'         : lr_history,
                'architecture'       : arch_str,          
                'in_features'        : X_train.shape[1],  
                'hidden_layer'       : num_nodes,         
                'out_features'       : 1,
                'dropout'            : dropout_rate,               
                'lr'                 : current_lr,
                'batch_size'         : batch_size,
            }, save_path)
 
            print(f"Ep {epoch+1:3d}: train={avg_train:.6f} "
                  f"val={avg_val:.6f} ✅ +{improvement:.6f}  "
                  f"lr={current_lr:.2e}  t={time.time()-t0:.1f}s")
            print(f"  💾 saved → {os.path.basename(save_path)}")
        else:
            patience_counter += 1
            print(f"Ep {epoch+1:3d}: train={avg_train:.6f} "
                  f"val={avg_val:.6f} ⚠ ({patience_counter}/{patience})  "
                  f"lr={current_lr:.2e}  t={time.time()-t0:.1f}s")
 
        prev_lr = current_lr
        scheduler.step(avg_val)
        new_lr = optimizer.param_groups[0]['lr']
        if new_lr < prev_lr:
            print(f"  📉 LR: {prev_lr:.2e} → {new_lr:.2e}")
 
        if patience_counter >= patience:
            print(f"\n{'='*60}")
            print(f"🛑 EARLY STOPPING "
                  f"epoch={epoch+1}  best={best_epoch+1}  "
                  f"best_val={best_val_loss:.6f}")
            print(f"{'='*60}")
            break
 
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, train_losses, val_losses, best_epoch, best_state

# ════════════════════════════════════════════════════════════════════════════
# RUN TRAINING 
# ════════════════════════════════════════════════════════════════════════════
bestmodel_dir = "/raid/nfs_storageIPB/thesis_JL/data/ANN_bestmodel_pth/"
save_path = os.path.join(bestmodel_dir,
    f'ANN_{source_id}_{num_nodes}_{dropout_rate}_{LR}.pth')
 
model, tl, vl, best_ep, _ = train_model(
    model, train_loader, val_loader,
    save_path)


────────────────────────────────────────────────────────────
Training ANN model arch=10→32_16→1 
Early stopping: patience=30, min_delta=1e-05
────────────────────────────────────────────────────────────


Ep   1: train=0.075032 val=0.023712 ✅ +inf  lr=3.00e-04  t=3.5s
  💾 saved → ANN_MPI-ESM1-2-HR_32_16_0.2_0.0003.pth


Ep   2: train=0.034389 val=0.018846 ✅ +0.004866  lr=3.00e-04  t=3.2s
  💾 saved → ANN_MPI-ESM1-2-HR_32_16_0.2_0.0003.pth


Ep   3: train=0.026182 val=0.016000 ✅ +0.002846  lr=3.00e-04  t=3.0s
  💾 saved → ANN_MPI-ESM1-2-HR_32_16_0.2_0.0003.pth


Ep   4: train=0.021623 val=0.014634 ✅ +0.001367  lr=3.00e-04  t=3.0s
  💾 saved → ANN_MPI-ESM1-2-HR_32_16_0.2_0.0003.pth


...

Ep  40: train=0.010039 val=0.011443 ⚠ (26/30)  lr=3.75e-05  t=3.0s


Ep  41: train=0.010034 val=0.011422 ⚠ (27/30)  lr=3.75e-05  t=3.0s


Ep  42: train=0.010038 val=0.011452 ⚠ (28/30)  lr=3.75e-05  t=2.8s


Ep  43: train=0.010030 val=0.011403 ⚠ (29/30)  lr=3.75e-05  t=3.0s


Ep  44: train=0.010002 val=0.011414 ⚠ (30/30)  lr=3.75e-05  t=2.9s

🛑 EARLY STOPPING epoch=44  best=14  best_val=0.010899


## Evalution

In [16]:
# ════════════════════════════════════════════════════════════════════════════
# CALCULATE METRICS
# ════════════════════════════════════════════════════════════════════════════
def calculate_metrics(model, data_loader, device,
                       inverse_fn, dataset_name="Dataset"):
    """
    Calculate evaluation metics for ANN output: scaler.inverse_transform_y
    """
    model.eval()
    all_preds, all_targets = [], []
 
    with torch.no_grad():
        for batch in data_loader:
            bX   = batch[0].to(device)
            by   = batch[1].to(device)
            pred = model(bX)
            all_preds  .append(pred.cpu().numpy())
            all_targets.append(by  .cpu().numpy())
 
    preds_s   = np.concatenate(all_preds)  .flatten()
    targets_s = np.concatenate(all_targets).flatten()
 
    preds_orig   = inverse_fn(preds_s)
    targets_orig = inverse_fn(targets_s)

    # Calculate all evaluation metrics in original unit (kWh)
    r2   = r2_score(targets_orig, preds_orig)
    corr, pval = stats.pearsonr(targets_orig, preds_orig)
    mae  = mean_absolute_error (targets_orig, preds_orig)
    rmse = np.sqrt(mean_squared_error(targets_orig, preds_orig))
    mse = mean_squared_error(targets_orig, preds_orig)
    mape = mean_absolute_percentage_error(targets_orig, preds_orig)
 
    return dict(dataset=dataset_name, r2=r2, correlation=corr,
                p_value=pval, mae=mae, rmse=rmse, mse=mse, mape=mape,
                predictions_scaled=preds_s, targets_scaled=targets_s,
                predictions_original=preds_orig, targets_original=targets_orig)

inv = scaler.inverse_transform_y 
train_m = calculate_metrics(model, train_loader, device, inv, "TRAIN")
val_m   = calculate_metrics(model, val_loader,   device, inv, "VAL")
test_m  = calculate_metrics(model, test_loader,  device, inv, "TEST")

def print_results(label, tm, vm, testm, best_ep):
    print(f"\n{'─'*80}")
    print(f"Best epoch: {best_ep+1}\n")
    print(f"Architecture: {arch_str}")
    print(f"{'─'*80}")
    print(f"  {'Metric':<22} {'TRAIN':>12} {'VAL':>12} {'TEST':>12}")
    print(f"  {'─'*58}")
    for key, fmt in [('r2','10.4f'), ('correlation','10.4f'),
                     ('mae','10.4f'), ('rmse','10.4f'), ('mse','10.4f'), ('mape','10.4f')]:
        print(f"  {key:<22} {tm[key]:{fmt}} {vm[key]:{fmt}} {testm[key]:{fmt}}")
print_results("Results",  train_m, val_m, test_m, best_ep)


────────────────────────────────────────────────────────────────────────────────
Best epoch: 14

Architecture: 10→32_16→1
────────────────────────────────────────────────────────────────────────────────
  Metric                        TRAIN          VAL         TEST
  ──────────────────────────────────────────────────────────
  r2                         0.5436     0.4952     0.5303
  correlation                0.7458     0.7348     0.7382
  mae                        0.0743     0.0767     0.0828
  rmse                       0.0938     0.0970     0.1018
  mse                        0.0088     0.0094     0.0104
  mape                       0.0688     0.0689     0.0787


## Save best model

In [17]:
# ════════════════════════════════════════════════════════════════════════════
# FIX 4 — Save metrics to file with dynamic architecture string
# ════════════════════════════════════════════════════════════════════════════
def save_metrics_file(path, tm, vm, testm, best_ep, total_ep):
    with open(path, 'w') as f:
        f.write(f"Number of PCs from climate vars: {n_pc}\n")
        f.write(f"Architecture                   : {arch_str}\n")  
        f.write(f"In-features                    : {X_train.shape[1]}\n")
        f.write(f"Dropout                        : {dropout_rate}\n")  
        f.write(f"Learning Rate                  : {LR}\n\n")
        f.write(f"Best epoch      : {best_ep+1}\n")
        f.write(f"Total epochs run: {total_ep}\n\n")
        f.write(f"{'Metric':<22} {'TRAIN':>12} {'VAL':>12} {'TEST':>12}\n")
        f.write(f"{'─'*58}\n")
        for key in ['r2', 'correlation', 'p_value', 'mae', 'rmse', 'mse', 'mape']:
            fmt = '12.4e' if key == 'p_value' else '12.4f'
            f.write(f"{key:<22} {tm[key]:{fmt}} {vm[key]:{fmt}} {testm[key]:{fmt}}\n")
    print(f"✅ Metrics saved → {os.path.basename(path)}")

metric_dir = "/raid/nfs_storageIPB/thesis_JL/data/ANN_evaluation_metric/"
save_metrics_file(
    os.path.join(
        metric_dir, f'metrics_10climvar_{source_id}_{X_train.shape[1]}_{dropout_rate}_1970-2014.txt'),
    train_m, val_m, test_m, best_ep, len(tl))

✅ Metrics saved → metrics_10climvar_MPI-ESM1-2-HR_10_0.2_1970-2014.txt


In [18]:
def save_metrics_excel(path, tm, vm, testm):
    """
    Save metrics to Excel file in a clean table format.
    """
    # Prepare metrics data
    metrics = ['r2', 'correlation', 'mae', 'rmse', 'mse', 'mape']
    data = {
        'Metric': metrics,
        'TRAIN': [tm[key] for key in metrics],
        'VAL': [vm[key] for key in metrics],
        'TEST': [testm[key] for key in metrics]
    }
    
    # Create DataFrame
    df = pd.DataFrame(data)
    
    # Save to Excel
    df.to_excel(path, index=False, sheet_name='Metrics')
    
    print(f"✅ Metrics saved → {os.path.basename(path)}")

save_metrics_excel(
    os.path.join(metric_dir, f'metrics_10climvar_{source_id}_{X_train.shape[1]}_{dropout_rate}_1970-2014.xlsx'),
    train_m, val_m, test_m
)

✅ Metrics saved → metrics_10climvar_MPI-ESM1-2-HR_10_0.2_1970-2014.xlsx


## Calculate prediction for full dataset

In [19]:
# Get original shape info
n_time, n_lat, n_lon = PV_target.shape

# Transform full dataset
X_full, y_full, valid_mask_full, grid_idx_full = extract_raw_data(
    PC_dict, pc_name, PV_target, lon_da, lat_da,
    moy_da, DEM_da, start_year=1970, end_year=2014)

# Scale X_full before prediction
X_full_scaled =  scaler.transform_X(X_full)

def predict_full_dataset_batched(model, X_full, device, batch_size=100000):
    """
    Generate predictions for all samples using batch processing
    
    Returns:
    --------
    numpy array: Predictions with shape (n_samples,) - 1D array
    """
    model.eval()
    predictions = []
    
    with torch.no_grad():
        for i in range(0, len(X_full), batch_size):
            batch = torch.FloatTensor(X_full[i:i+batch_size]).to(device)
            pred_batch = model(batch)
            
            # FIX: Squeeze to remove extra dimension before converting to numpy
            pred_batch = pred_batch.cpu().numpy()
            if pred_batch.ndim > 1 and pred_batch.shape[1] == 1:
                pred_batch = pred_batch.squeeze()  # or pred_batch.reshape(-1)
            
            predictions.append(pred_batch)
    
    # Concatenate along axis 0
    return np.concatenate(predictions)  # This will now be 1D


# Calculate prediction for full dataset
predictions_scaled_full = predict_full_dataset_batched(
    model, X_full_scaled, device)

time_coord = PV_target.sel(time=slice('1970','2014')).time.values
lat_coord = PV_target.sel(time=slice('1970','2014')).lat.values
lon_coord = PV_target.sel(time=slice('1970','2014')).lon.values

def inverse_transform_predictions(predictions_scaled, pv_original, valid_mask, time_coords, lat_coords, lon_coords):
    """
    Convert scaled predictions back to original values and reshape to 3D xarray DataArray
    
    Parameters:
    -----------
    predictions_scaled : numpy array or torch tensor
        Scaled predictions from the model (flattened)
    pv_original : xarray DataArray
        Original PV data with NaN values (to get mean and std for inverse transform)
    valid_mask : numpy array
        Boolean mask of valid samples used during training (flattened)
    time_coords, lat_coords, lon_coords : arrays
        Original coordinate values
    
    Returns:
    --------
    xarray.DataArray with original values in 3D shape (time, lat, lon)
    """
    
    # 1. Convert to numpy if it's a torch tensor
    if hasattr(predictions_scaled, 'cpu'):
        predictions_scaled = predictions_scaled.cpu().numpy()
    
    # 2. Squeeze the predictions to remove any extra dimensions
    # This handles both (n_samples, 1) and (n_samples,) shapes
    predictions_scaled = np.squeeze(predictions_scaled)
    
    print(f"\nPredictions_scaled shape after squeeze: {predictions_scaled.shape}")
    
    # 3. Inverse transform: scaler.inverse_transform_y
    predictions_original = scaler.inverse_transform_y(predictions_scaled)
    
    print(f"\nPredictions_original shape: {predictions_original.shape}")
    print(f"Valid mask shape: {valid_mask.shape}")
    print(f"Valid mask sum: {valid_mask.sum()}")
    print(f"Predictions_original length: {len(predictions_original)}")
    
    # 5. Create empty array with original shape filled with NaN
    n_time, n_lat, n_lon = pv_original.shape
    n_grid = n_lat * n_lon
    n_total = n_time * n_grid
    
    print(f"\nOriginal shape: ({n_time}, {n_lat}, {n_lon})")
    print(f"Total cells: {n_total}")
    print(f"Valid samples: {valid_mask.sum()}")
    
    # Create flattened array with NaN
    predictions_flat = np.full(n_total, np.nan)
    
    # 6. Fill valid positions with predictions
    # Ensure both arrays are 1D
    if predictions_original.ndim > 1:
        predictions_original = predictions_original.flatten()
    
    predictions_flat[valid_mask] = predictions_original
    
    # 7. Reshape to 3D (time, lat, lon)
    predictions_3d = predictions_flat.reshape(n_time, n_lat, n_lon)
    
    # 8. Create xarray DataArray
    predictions_da = xr.DataArray(
        predictions_3d,
        dims=['time', 'lat', 'lon'],
        coords={
            'time': time_coords,
            'lat': lat_coords,
            'lon': lon_coords
        },
        name='pv_downscaled_predicted',
        attrs={
            'description': 'Downscaled PV predictions from neural network',
            'units': PV_target.attrs.get('units', 'original units'),
            'model': f'{source_id}',
            'PCs_features' : f'{n_pc}'
        }
    )
    
    # 9. Quick validation
    print(f"\nPredictions 3D shape: {predictions_3d.shape}")
    print(f"NaN count in predictions: {np.isnan(predictions_3d).sum()}")
    print(f"Predictions range: [{predictions_3d.min():.4f}, {predictions_3d.max():.4f}]")
    return predictions_da

# Now inverse transform to original values and reshape
predictions_3d_da = inverse_transform_predictions(
    predictions_scaled=predictions_scaled_full,
    pv_original=PV_target,
    valid_mask=valid_mask_full,
    time_coords=time_coord,
    lat_coords=lat_coord,
    lon_coords=lon_coord)


Filtering time dimension: 1970 to 2014
Original time points: 540
Selected time points: 540

Data dimensions after filtering:
  Time points: 540
  Grid points: 7000 (50 × 140)
  Climate features: 5

Predictions_scaled shape after squeeze: (751140,)

Predictions_original shape: (751140,)
Valid mask shape: (3780000,)
Valid mask sum: 751140
Predictions_original length: 751140

Original shape: (540, 50, 140)
Total cells: 3780000
Valid samples: 751140

Predictions 3D shape: (540, 50, 140)
NaN count in predictions: 3028860
Predictions range: [nan, nan]


In [20]:
# Get attributes from all coordinate dimensions
time_att = PV_target.time.attrs 
lon_att = PV_target.lon.attrs
lat_att = PV_target.lat.attrs

# Define a function to assign dataarray attributes for bias-corrected data
def assign_dim_attrs(da):
    da['time'] = da.time.assign_attrs(time_att)
    da['lat'] = da.lat.assign_attrs(lat_att)
    da['lon'] = da.lon.assign_attrs(lon_att)
    return da

# Define dataarray's attributes
da_attrs = {
    'standard_name': 'pv_energy_potential',
    'long_name': 'Monthly PV energy potential',
    'units': 'kWh m-2 month-1',
    }
ds_attrs = {
    'title': 'Statistical Downscaling output based on ANN approach',
    'authors': 'Jassica Listyarin',
    'institution': 'Department of Geophysics and Meteorology, Faculty of Mathematics and Natural Sciences, IPB University',
    'table_id': 'mon',
}

# Assign attributes for bias-corrected baseline data
VAR = 'PVout'
ANN_attrs = assign_dim_attrs(predictions_3d_da)
ANN_da_attrs = ANN_attrs.assign_attrs(da_attrs)
ANN_ds = ANN_da_attrs.to_dataset(dim=None, name=f'{VAR}')
ANN_ds= ANN_ds.assign_attrs(ds_attrs)

In [21]:
# Let's save ANN output to netcdf file
outdir_ann = f'/raid/nfs_storageIPB/thesis_JL/data/ANN_output/'# Define file's configurations
path_ann = Path(outdir_ann)/source_id
try:
    path_ann.mkdir(parents=True, exist_ok=True)
    print(f"Directory '{source_id}' is created or already exists")
except Exception as e:
    print(f"Failed to create directory: {str(e)}")
    raise
period = '197001-201412'
table_id = 'mon'

def save_dataset(ds):
    """Helper function to save datasets into netcdf format with error handling"""
    try:
        filename = f"ANN_10climvar_{VAR}_{table_id}_{source_id}_historical_{period}.nc"
        output_path = path_ann / filename
        # Add another attributes
        ds.attrs['experiment_id'] = 'historical'
        ds.attrs['frequency'] = table_id
        ds.attrs['mip_era'] = 'CMIP6'
        ds.attrs['source_id'] = source_id

        print(f"Attempting to save to: {output_path}")

        # Load and write in one operation to minimize memory usage
        ds.to_netcdf(
            output_path,
            format="NETCDF4",
            engine="netcdf4",
            encoding={VAR: {"dtype": "float32"}, 
                  'time': {'dtype': 'float64', 
                           'units': 'days since 1850-01-01 00:00:00'}},  # Use variable name from VAR
            unlimited_dims=['time']
        )
        print(f"Successfully saved: {filename}")
    except Exception as e:
        print(f"Failed to save {filename}: {str(e)}")
        raise

# Save bias-corrected data to new netcdf file
save_dataset(ANN_ds)

Directory 'MPI-ESM1-2-HR' is created or already exists
Attempting to save to: /raid/nfs_storageIPB/thesis_JL/data/ANN_output/MPI-ESM1-2-HR/ANN_10climvar_PVout_mon_MPI-ESM1-2-HR_historical_197001-201412.nc
Successfully saved: ANN_10climvar_PVout_mon_MPI-ESM1-2-HR_historical_197001-201412.nc


In [ ]:
def flatten_valid_1d(arr_target, arr_ann, arr_anncdf):
    """
    Flatten dua array (1D, 2D, atau 3D) ke 1D,
    buang semua posisi NaN secara bersamaan.

    Berlaku untuk semua bentuk input:
      - [n_samples]          ← output ANN (sudah 1D)
      - [n_time × n_grid]    ← setelah reshape xarray
      - [n_time × lat × lon] ← xarray DataArray langsung

    Returns
    -------
    a_1d, b_1d, c_1d : np.ndarray [n_valid] — keduanya 1D, tanpa NaN
    """
    target_flat = np.asarray(arr_target).flatten()
    ann_flat = np.asarray(arr_ann).flatten()
    anncdf_flat = np.asarray(arr_anncdf).flatten()
    
    # Buang posisi NaN di salah satu atau keduanya
    mask   = ~np.isnan(target_flat) & ~np.isnan(ann_flat) & ~np.isnan(anncdf_flat)
    
    return target_flat[mask], ann_flat[mask], anncdf_flat[mask]

# Flatten ke 1D dan buang NaN
TARGET_train_, ANN_train_, ANNCDF_train_ = flatten_valid_1d(
    TARGET_dict['train'], 
    ANN_dict['train'],
    ANNCDF_dict['train'])

TARGET_val_, ANN_val_, ANNCDF_val_ = flatten_valid_1d(
    TARGET_dict['val'], 
    ANN_dict['val'],
    ANNCDF_dict['val'])

TARGET_test_, ANN_test_, ANNCDF_test_ = flatten_valid_1d(
    TARGET_dict['test'], 
    ANN_dict['test'],
    ANNCDF_dict['test'])


In [ ]:
def hitung_metrik_1d(target, pred, label=""):
    """
    Hitung metrik evaluasi — INPUT APAPUN → flatten 1D.

    Parameters
    ----------
    target : array-like — 1D 
    pred   : array-like — 1D
    label  : str

    Returns
    -------
    dict metrik scalar
    """
    
    metrik = {
        'label'       : label,
        'n_samples'   : len(target),
        'r2'          : r2_score(target, pred),
        'correlation' : stats.pearsonr(target, pred).statistic,
        'mae'         : mean_absolute_error(target, pred),
        'mse'         : mean_squared_error(target, pred),
        'rmse'        : np.sqrt(mean_squared_error(target, pred)),
        'mape'        : mean_absolute_percentage_error(
                            target, pred) * 100,  # in %
    }
    return metrik

combination = [
    (TARGET_train_, ANN_train_,    'ANN',    'TRAIN'),
    (TARGET_val_,   ANN_val_,      'ANN',    'VAL'),
    (TARGET_test_,  ANN_test_,     'ANN',    'TEST'),
    (TARGET_train_, ANNCDF_train_, 'ANN+QM', 'TRAIN'),
    (TARGET_val_,   ANNCDF_val_,   'ANN+QM', 'VAL'),
    (TARGET_test_,  ANNCDF_test_,  'ANN+QM', 'TEST'),
]

print("Evaluation Metrics")
print("═"*65)
eval_global_ = []
for target, pred, model, split in combination:
    m = hitung_metrik_1d(
            target, pred,
            label=f"{model} — {split}"
        )
    eval_global_.append(m)
    print(f"  {m['label']:<22}: "
          f"R²={m['r2']:.4f}  "
          f"Correlation={m['correlation']:.4f} "
          f"MAE={m['mae']:.4f} "
          f"RMSE={m['rmse']:.4f}  "
          f"MAPE={m['mape']:.2f}%  "
          f"(n={m['n_samples']:,})")

df_g = pd.DataFrame(eval_global_).set_index('label')